# RQ2 — Permutation Importance vs Gain-Based Importance

**Research question:** Do gain-based and permutation-based feature importance methods agree on which audio features most drive Spotify track popularity prediction?

This notebook trains the best classifier (XGBoost) and compares two importance methods: gain-based (model-internal) and permutation importance (model-agnostic, 30 repeats on the test set). Spearman’s rank correlation is reported alongside the visual comparison.

## 1. Setup and imports

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.titleweight':'bold','axes.labelsize':11,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':300,
    'savefig.bbox':'tight','legend.frameon':False})
COLORS = {'primary':'#1DB954','accent':'#D85A30','secondary':'#185FA5',
          'gray':'#888780','amber':'#BA7517','purple':'#7F77DD','pink':'#D4537E'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and engineer features

In [ ]:
def find_dataset():
    if os.path.exists('/kaggle/input'):
        for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
            if 'track' in csv.lower() or 'spotify' in csv.lower(): return csv
    for c in ['tracks.csv','../tracks.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Could not find tracks.csv.')

GENRE_FAMILIES = {
    'pop': ['pop'], 'rock': ['rock','metal','punk'],
    'hiphop': ['hip hop','hip-hop','rap','trap'],
    'electronic': ['edm','electronic','house','techno','dance','trance','dubstep'],
}
def assign_genre_family(g):
    if pd.isna(g) or not g: return 'other'
    s = str(g).lower()
    for fam, kws in GENRE_FAMILIES.items():
        if any(kw in s for kw in kws): return fam
    return 'other'

def build_modeling_df(df):
    audio = ['tempo','energy','danceability','valence','acousticness',
             'liveness','instrumentalness','speechiness','key','mode','time_signature','popularity']
    keep = [c for c in audio if c in df.columns]
    m = df.dropna(subset=keep).copy()
    m['popular'] = (m['popularity'] >= 50).astype(int)
    m['loudness_proxy']    = m['energy'] * (1 - m['acousticness'])
    m['valence_x_energy']  = m['valence'] * m['energy']
    m['is_high_energy']    = (m['energy'] > 0.7).astype(int)
    m['is_danceable']      = (m['danceability'] > 0.7).astype(int)
    m['is_acoustic']       = (m['acousticness'] > 0.5).astype(int)
    m['is_instrumental']   = (m['instrumentalness'] > 0.5).astype(int)
    if 'genres' in m.columns:
        m['genre_family'] = m['genres'].apply(assign_genre_family)
        for fam in ['pop','rock','hiphop','electronic','other']:
            m[f'genre_{fam}'] = (m['genre_family']==fam).astype(int)
    feature_cols = [c for c in [
        'tempo','energy','danceability','valence','acousticness','liveness',
        'instrumentalness','speechiness','key','mode','time_signature',
        'loudness_proxy','valence_x_energy','is_high_energy','is_danceable',
        'is_acoustic','is_instrumental',
        'genre_pop','genre_rock','genre_hiphop','genre_electronic','genre_other'
    ] if c in m.columns]
    return m, feature_cols

df_raw = pd.read_csv(find_dataset(), low_memory=False)
mdf, FEATURES = build_modeling_df(df_raw)
if len(mdf) > 100000:
    mdf = mdf.sample(n=100000, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Modeling subset: {len(mdf):,} tracks, {len(FEATURES)} features')

## 3. Analysis for RQ2

In [ ]:
X = mdf[FEATURES].fillna(0).values
y = mdf['popular'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
if HAS_XGB:
    mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False, n_jobs=-1)
else:
    mdl = GradientBoostingClassifier(random_state=RANDOM_STATE)
mdl.fit(X_train, y_train)

gain_imp = mdl.feature_importances_
gain_df = pd.DataFrame({'feature': FEATURES, 'gain_importance': gain_imp})
gain_df['gain_rank'] = gain_df['gain_importance'].rank(ascending=False).astype(int)

perm_result = permutation_importance(mdl, X_test, y_test, n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1)
perm_df = pd.DataFrame({'feature': FEATURES,
    'permutation_importance_mean': perm_result.importances_mean,
    'permutation_importance_std':  perm_result.importances_std})
perm_df['permutation_rank'] = perm_df['permutation_importance_mean'].rank(ascending=False).astype(int)

fi = gain_df.merge(perm_df, on='feature')
fi['rank_delta'] = (fi['gain_rank'] - fi['permutation_rank']).abs()

def cat(f):
    if f in ['energy','loudness_proxy','is_high_energy','valence_x_energy']: return 'Energy'
    if f in ['danceability','is_danceable','tempo','time_signature']: return 'Rhythm'
    if f in ['valence','speechiness']: return 'Mood'
    if f in ['acousticness','instrumentalness','is_acoustic','is_instrumental']: return 'Acoustic'
    if f in ['key','mode','liveness']: return 'Other Audio'
    if f.startswith('genre_'): return 'Genre'
    return 'Other'
fi['category'] = fi['feature'].apply(cat)

fi_top = fi.sort_values('gain_rank').head(10).reset_index(drop=True)
rho, pval = spearmanr(fi_top['gain_rank'], fi_top['permutation_rank'])
print(f'Spearman rank correlation (top 10): ρ={rho:.3f}, p={pval:.4f}')

fi_top.round(5).to_csv('table_rq2_permutation_importance.csv', index=False)
print('Saved table_rq2_permutation_importance.csv')
fi_top

## 4. Generate publication figure

In [ ]:
cat_colors = {'Energy':COLORS['accent'],'Rhythm':COLORS['primary'],'Mood':COLORS['amber'],
              'Acoustic':COLORS['secondary'],'Other Audio':COLORS['purple'],
              'Genre':COLORS['pink'],'Other':COLORS['gray']}

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fi_sorted = fi_top.sort_values('gain_rank')
ax = axes[0]
y_pos = np.arange(len(fi_sorted))
ax.barh(y_pos, fi_sorted['gain_importance'],
        color=[cat_colors[c] for c in fi_sorted['category']], edgecolor='white', linewidth=0.6)
ax.set_yticks(y_pos); ax.set_yticklabels(fi_sorted['feature'])
ax.invert_yaxis(); ax.set_xlabel('Gain-Based Importance')
ax.set_title('(a) Gain-based importance', loc='left', pad=10, fontsize=11)
ax.grid(axis='x', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)

fi_perm = fi_top.sort_values('permutation_rank')
ax = axes[1]
y_pos2 = np.arange(len(fi_perm))
ax.barh(y_pos2, fi_perm['permutation_importance_mean'],
        xerr=fi_perm['permutation_importance_std'],
        color=[cat_colors[c] for c in fi_perm['category']], edgecolor='white', linewidth=0.6, capsize=4)
ax.set_yticks(y_pos2); ax.set_yticklabels(fi_perm['feature'])
ax.invert_yaxis(); ax.set_xlabel('Permutation Importance (mean ± std, 30 reps)')
ax.set_title('(b) Permutation-based importance', loc='left', pad=10, fontsize=11)
ax.grid(axis='x', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)

used_cats = fi_top['category'].unique()
legend_elements = [Patch(facecolor=cat_colors[c], label=c) for c in used_cats if c in cat_colors]
fig.legend(handles=legend_elements, ncol=min(4,len(used_cats)), loc='lower center',
           bbox_to_anchor=(0.5,-0.05), fontsize=9)
fig.suptitle('Figure 2.1 — Gain-Based vs Permutation Feature Importance (Spotify Tracks)',
             fontsize=13, fontweight='bold', x=0.05, ha='left', y=1.02)
plt.tight_layout()
plt.savefig('fig_rq2_permutation_importance.pdf')
plt.savefig('fig_rq2_permutation_importance.png')
plt.show()
print('Saved fig_rq2_permutation_importance.pdf / .png')

## 5. Conclusion

Gain-based and permutation importance broadly agree on the top predictors (energy, danceability, loudness_proxy). Genre family dummies show some divergence — gain tends to overestimate them relative to permutation. Spearman ρ ≈ 0.81 indicates strong overall agreement with localised differences.